# Generation using Retrieved Data


## Setup OpenAI API


In [3]:
import os

import azure.identity
import dotenv
import openai

# Set up OpenAI client based on environment variables
dotenv.load_dotenv()
print("Loaded environment variables from .env file")
print(dotenv.dotenv_values(".env"))
AZURE_OPENAI_SERVICE = os.getenv("AZURE_OPENAI_SERVICE")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
AZURE_OPENAI_TEXT_COMPLETION_MODEL = os.getenv("AZURE_OPENAI_TEXT_COMPLETION_MODEL")

azure_credential = azure.identity.AzureDeveloperCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID"))
token_provider = azure.identity.get_bearer_token_provider(
    azure_credential, "https://cognitiveservices.azure.com/.default"
)
openai_client = openai.AzureOpenAI(
    api_version="2024-06-01",
    azure_endpoint=f"https://{AZURE_OPENAI_SERVICE}.openai.azure.com",
    azure_ad_token_provider=token_provider,
)

Loaded environment variables from .env file
OrderedDict({'AZURE_OPENAI_SERVICE': 'ai-prateek4732ai561893487136', 'AZURE_OPENAI_TEXT_COMPLETION_MODEL': 'gpt-4.1-nano', 'AZURE_OPENAI_EMBEDDING_MODEL': 'text-embedding-3-small', 'AZURE_TENANT_ID': '06f18712-6c3a-4b61-9475-bf2c226971b3'})


### Recommendation System


In [5]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import faiss

# 1. Load movie data
with open("movies.json", "r") as f:
    data = json.load(f)

movies = data["movies"]
titles = [movie["title"] for movie in movies]
texts = [f"{movie['title']} {movie['plot']}" for movie in movies]

# 2. Load FAISS index
index = faiss.read_index("movie_embeddings.index")
num_vectors = index.ntotal
dimension = index.d

# 3. Reconstruct embeddings from index
embeddings = np.vstack([index.reconstruct(i) for i in range(num_vectors)])

# 4. Generate query embedding
query_text = "A space adventure with aliens and distant planets"
# query_text = "Story of a warrior who fights for his kingdom"
response = openai_client.embeddings.create(model="text-embedding-ada-002", input=[query_text])
query_vector = np.array(response.data[0].embedding).astype("float32").reshape(1, -1)
print(f"[User] {query_text}")

# 5. Compute cosine similarity and select top k results
similarities = cosine_similarity(query_vector, embeddings)[0]
df = pd.DataFrame({"Title": titles, "Similarity": similarities, "Index": np.arange(len(titles))}).sort_values(
    "Similarity", ascending=False
)
k = 5
top_movies = df.head(k)

print("Top movies retrieved:", top_movies[["Title", "Similarity",]])

# 6. Build descriptions for LLM prompt
descriptions = []
for _, row in top_movies.iterrows():
    movie = movies[int(row["Index"])]
    desc = (
        f"Title: {movie['title']}\nYear: {movie['year']}\nGenres: {', '.join(movie['genres'])}\nPlot: {movie['plot']}"
    )
    descriptions.append(desc)

movie_descriptions = "\n\n".join(descriptions)

# 7. Build prompt and call OpenAI chat LLM
prompt = f"""
A user is looking for a movie recommendation similar to: "{query_text}".

Here are the top recommended movies:

{movie_descriptions}

Based on these, write a friendly recommendation message in 2-3 sentences, highlighting which movie the user should watch first and why.
"""

response = openai_client.chat.completions.create(
    model=AZURE_OPENAI_TEXT_COMPLETION_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful movie recommendation assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
    max_tokens=500,
)

recommendation = response.choices[0].message.content

# Final output
print("\n[Assistant] Movie recommendations:\n")
print(recommendation)

[User] A space adventure with aliens and distant planets
Top movies retrieved:                                  Title  Similarity
12                        Interstellar    0.871398
18                              Avatar    0.858849
19  Star Wars: Episode IV - A New Hope    0.828639
9                           The Matrix    0.817927
17                        The Avengers    0.814472

[Assistant] Movie recommendations:

I recommend starting with *Interstellar* because it offers a thrilling space adventure with captivating visuals and a compelling story about exploring distant planets to save humanity. If you enjoy exploring the cosmos and the mysteries of space, this film is a perfect choice. Plus, it beautifully combines adventure, drama, and sci-fi elements that align well with your interests!
